# Preparar y estructurar datos con Python: ejemplos de las lecciones

Ejemplos de la explicación de las lecciones 2, 3 y 4. Los laboratorios guiados y los retos están en `pandas_labs.ipynb`.

Ejecuta las celdas en orden: cada lección usa las variables de la anterior.

## Preparación: carga de los datos crudos

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

RUTA = Path('../data/raw')
SALIDA = Path('../data/processed')

orders      = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products    = pd.read_csv(RUTA / 'products.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')

print(pd.__version__)
print(orders.shape, products.shape, order_prods.shape, aisles.shape, departments.shape)
# Esperado: (478967, 6) (49694, 4) (4545007, 4) (134, 2) (21, 2)

3.0.6
(478967, 6) (49694, 4) (4545007, 4) (134, 2) (21, 2)


# Lección 2: Limpieza y estandarización de datos crudos

Orden de trabajo: duplicados, tipos, nulos, texto, duplicados implícitos y verificación.

### Duplicados exactos: detectar con `duplicated`

In [12]:
duplicados = orders[orders.duplicated(keep=False)].sort_values('order_id')
print(len(duplicados))
duplicados[['order_id', 'order_dow', 'order_hour_of_day']].head(6)
# Esperado: 30 filas, todas con order_dow = 3 y order_hour_of_day = 2

30


,order_id,order_dow,order_hour_of_day
354993,391768,3,2
371905,391768,3,2
119251,408114,3,2
321100,408114,3,2
394347,467134,3,2
250626,467134,3,2


### Duplicados exactos: eliminar con `drop_duplicates`

In [13]:
orders = orders.drop_duplicates().reset_index(drop=True)
print(orders.shape, orders['order_id'].is_unique)
# Esperado: (478952, 6) True

(478952, 6) True


### Tipos de dato con `astype`

In [14]:
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('Después:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')
# Esperado: Antes: 23.0 MB / Después: 7.2 MB

Antes: 23.0 MB
Después: 7.2 MB


### Nulos: comprobar la causa antes de decidir

In [15]:
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())
# Esperado: True -> cada nulo es un primer pedido: se conserva

True


In [16]:
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())
# Esperado: una combinación (100, 21) con 1258 productos y luego 0

aisle_id  department_id
100       21               1258
Name: count, dtype: int64
0


| Columna | Causa | Decisión |
| --- | --- | --- |
| `days_since_prior_order` | Primer pedido del cliente | Conservar el nulo |
| `product_name` | Producto sin datos de catálogo | Rellenar con `'Unknown'` |
| `add_to_cart_order` | Se investiga en el Reto 2 | Conservar el nulo |

### Texto con los métodos `.str`

In [17]:
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())
# Esperado: Espacios dobles: 403 / Espacio invisible: 16 / 0 0

Espacios dobles: 403
Espacio invisible: 16
0 0


### Duplicados implícitos

In [18]:
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))
# Esperado: 141 / 268 de 280 -> conservar y documentar; agrupar por product_id

141
268 de 280


### Limpieza reutilizable con funciones

In [19]:
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)
# Esperado: (478952, 6)

(478952, 6)


### Verificación de toda la limpieza

In [20]:
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'
# Si la celda no muestra nada, la limpieza está completa